# Put Your Enterprise Knowledge to Work in Claude Desktop
Give Claude Desktop secure, governed access to the knowledge in your own database. This Notebook connects an AIDB agent tool, in this case a tax-document search, to Claude Desktop, so people can ask questions in plain language and get answers grounded in your documents, with sources.

Technologies used:

- **EDB Postgres AI (AIDB)**: pipelines, knowledge bases, and agent tools
- **Model Context Protocol (MCP)**: the open standard that lets AI apps use your tools
- **Claude Desktop**: the AI assistant your users work in
- **PostgreSQL role-based access control**: the agent only sees what you grant it
- **Docker and AWS**: the same setup runs locally or in the cloud

## How it fits together

```
Claude Desktop  --(stdio)-->  mcp-curl-bridge.sh  --(HTTPS + login)-->  AIDB MCP endpoint in Postgres
                                                                              |
                                                          runs as the Postgres role "mcp_agent"
                                                                              |
                                                       mcp_search_tax_docs tool --> tax PDF knowledge base
```

When you ask a question in Claude Desktop, Claude decides to call the `mcp_search_tax_docs` tool. The request travels over HTTPS to Postgres, which runs the tool *as the `mcp_agent` role*. So the only data Claude can reach is data you have granted to that role.

## What you will do

1. **Prepare the MCP server**: create a restricted database role, a login rule for it, a TLS certificate, and the server settings.
2. **Build the knowledge base**: load two IRS PDFs and create an AI pipeline that makes them searchable.
3. **Create the SQL tool** that Claude will call.
4. **Connect Claude Desktop**: download a small bridge script and add a server entry to Claude's configuration.
5. **Validate** by asking Claude a question that can only be answered from the PDFs.

## Requirements

1. **macOS**: the Claude Desktop steps in this notebook use macOS paths and commands (for example `~/.local/bin`).
2. **Claude Desktop**: installed on that Mac and signed in.
3. **Deployed in either Docker or AWS**: the notebook runs on the machine that hosts Postgres and AIDB, either as a local Docker container or on an AWS instance. The port used by the MCP server (8765 by default) must be reachable from your Mac. For AWS, that means allowing it in the instance's security group.

## Set variables

This demo can run locally in Docker or on AWS. The cell works out which one automatically:

- It asks the AWS instance metadata service for the machine's public hostname or IP address. If that works, you are on AWS and `MCP_HOST` is set to that address.
- If it does not respond, the notebook assumes a local Docker setup and uses `localhost`.

`MCP_HOST` is used later for the TLS certificate and for the URL in Claude's configuration. `MCP_PORT` is the port the MCP server listens on.

Also set the MCP password. `MCP_PASSWORD` is the password for the database role Claude will log in as. The default value, `changeme`, is only a placeholder, so **change it** to something strong before running the rest of the notebook.

In [ ]:
import urllib.request

def aws_public_host(timeout=1):
    try:
        req = urllib.request.Request(
            "http://169.254.169.254/latest/api/token", method="PUT",
            headers={"X-aws-ec2-metadata-token-ttl-seconds": "60"})
        token = urllib.request.urlopen(req, timeout=timeout).read().decode()
        for path in ("public-hostname", "public-ipv4"):
            req = urllib.request.Request(
                f"http://169.254.169.254/latest/meta-data/{path}",
                headers={"X-aws-ec2-metadata-token": token})
            val = urllib.request.urlopen(req, timeout=timeout).read().decode()
            if val:
                return val
    except Exception:
        pass
    return None

MCP_HOST = aws_public_host() or "localhost"
if MCP_HOST == "localhost":
    MCP_PORT = "8765"
else:
    #MCP_PORT = "18765"
    MCP_PORT = "8765"

print("Running on AWS" if MCP_HOST != "localhost" else "Running locally", "->", MCP_HOST)

MCP_PASSWORD = "changeme"

## Connect to Postgres

Creates a SQLAlchemy engine for the `aidb` database (as `postgres` over the local socket) and attaches the `%sql` magics to it. `pool_pre_ping=True` checks connections before reuse, and named parameters are enabled so `%%sql` cells can use Python variables such as `:MCP_PASSWORD`.

In [ ]:
from sqlalchemy import create_engine

PGUSER = "postgres"
PGDATABASE = "aidb"
PGSOCKET = "/run/postgresql"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}",
    pool_pre_ping=True
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## MCP: Create Database Role

Create a dedicated Postgres role, `mcp_agent`, that Claude Desktop will log in as. Using a separate role is the heart of the governance story: **Claude can do only what this role is allowed to do**.

- The role is created if it does not exist, or its password is updated if it does, so the cell can be re-run.
- `aidb_users` is a built-in AIDB role that allows use of AIDB functions and tools. Granting it to `mcp_agent` gives the role what it needs to run AIDB tools and nothing more.

Later steps grant this role read access to the specific tables the tool needs.

In [ ]:
%%sql

DO $$
BEGIN
   IF NOT EXISTS (SELECT FROM pg_catalog.pg_roles WHERE rolname = 'mcp_agent') THEN
      CREATE ROLE mcp_agent WITH LOGIN PASSWORD :MCP_PASSWORD;
   ELSE
       ALTER ROLE mcp_agent WITH PASSWORD :MCP_PASSWORD;
   END IF;
END
$$;

GRANT aidb_users TO mcp_agent; --built-in AIDB role 


## MCP: Configure password

This cell adds a rule to `pg_hba.conf`, the Postgres client-authentication file. The line `local all mcp_agent scram-sha-256` requires the `mcp_agent` role to prove its identity with a password when it connects to the local server.

- The rule is added at the top of the file so it is matched first, and only if it is not already there.
- `pg_reload_conf()` makes Postgres re-read the file without a restart.

In [ ]:
%%bash
HBA=/var/lib/pgsql/18/data/pg_hba.conf
LINE='local all mcp_agent scram-sha-256'

if grep -qxF "$LINE" "$HBA"; then
  echo "pg_hba.conf entry already present"
else
  sed -i "1i $LINE" "$HBA"
  echo "pg_hba.conf entry added"
fi

/usr/pgsql-18/bin/psql -d aidb -c "SELECT pg_reload_conf();"

## MCP: Create SSL Cert

Claude Desktop talks to the MCP server over HTTPS, so the server needs a TLS certificate. This cell uses `openssl` to generate a **self-signed** certificate and private key:

- A 2048-bit RSA key, valid for 365 days.
- The certificate names the host (`MCP_HOST`) and `127.0.0.1`, so it matches the address clients use.
- `chmod 600` limits the private key so that only its owner can read it.

A self-signed certificate is fine for a demo, but clients cannot verify it, which is why the later `curl` commands and the bridge script skip certificate checks. In production, use a certificate issued by a trusted certificate authority.

In [ ]:
!openssl req -x509 -newkey rsa:2048 -nodes -days 365 \
  -keyout /var/lib/pgsql/mcp-server.key \
  -out /var/lib/pgsql/mcp-server.crt \
  -subj "/CN={MCP_HOST}" \
  -addext "subjectAltName=DNS:{MCP_HOST},IP:127.0.0.1"
!chmod 600 /var/lib/pgsql/mcp-server.key
!ls -l /var/lib/pgsql/mcp-server.*

## MCP: Database settings

AIDB can serve its tool catalog (plus a built-in `execute_sql` tool) to external MCP clients over HTTP.
Every call runs as the Postgres role the client authenticates with, so Postgres grants decide what it can do.

These settings turn the feature on and configure it:

| Setting | Meaning |
|---|---|
| `edb.endpoints_mcp_enabled` | Turn the MCP endpoint on |
| `edb.endpoints_mcp_host` | Address to listen on. `0.0.0.0` means all network interfaces, so restrict access with a firewall or security group |
| `edb.endpoints_mcp_tls_cert` / `_tls_key` | The certificate and key created in the previous step |
| `edb.endpoints_mcp_database` | Which database the tools run against (`aidb`) |
| `edb.endpoints_mcp_port` | The port to listen on |

They are saved with `ALTER SYSTEM` and take effect when Postgres restarts. A restart is already planned in the next section, so it will apply these settings too.

In [ ]:
%%sql
ALTER SYSTEM SET edb.endpoints_mcp_enabled = on;
ALTER SYSTEM SET edb.endpoints_mcp_host = '0.0.0.0';
ALTER SYSTEM SET edb.endpoints_mcp_tls_cert = '/var/lib/pgsql/mcp-server.crt';
ALTER SYSTEM SET edb.endpoints_mcp_tls_key = '/var/lib/pgsql/mcp-server.key';
ALTER SYSTEM SET edb.endpoints_mcp_database = 'aidb';
ALTER SYSTEM SET edb.endpoints_mcp_port = {{MCP_PORT}};


# Knowledge Base on PDF documents
Upload PDF documents for the Knowledge Base and a SQL Tool.

We will parse the PDF files, chunk them into smaller sections, and then create embeddings for each chunk. This improves the accuracy of finding information found in the PDF documents. This is all done with an AI Pipeline.

The steps are: **storage location, volume, files, pipeline, search, SQL tool.**

## Create directory storage location

This example uses a local file location. The cell creates the folder on the database server where the PDFs will be stored.

In [ ]:
!mkdir -p /var/lib/pgsql/pgfs/pdfs

## Register path in database

AIDB reads files through **PGFS**, a Postgres file-system layer. For safety, Postgres only reads local folders that an administrator has explicitly allowed. This cell adds the folder to the `pgfs.allowed_local_fs_paths` allow-list, a colon-separated list, and keeps the folders already on it (other demos read their files from `/tmp/pgfs`). Like the MCP settings, it is saved with `ALTER SYSTEM` and takes effect after the restart.

The demo uses its own names (`mcp_pdfs`, `mcp_pdf_volume`, `mcp_pdf_kb`, `mcp_search_tax_docs`), so it does not replace the knowledge base of the RAG over Documents demo.

In [ ]:
MCP_PDF_ROOT = "/var/lib/pgsql/pgfs"

# load pgfs so that its setting can be read
%sql SELECT pgfs.pgfs_version();
result = %sql SHOW pgfs.allowed_local_fs_paths
allowed = [p for p in result[0][0].split(":") if p]
if MCP_PDF_ROOT in allowed:
    print("Already allowed:", ":".join(allowed))
else:
    allowed_paths = ":".join(allowed + [MCP_PDF_ROOT])
    %sql ALTER SYSTEM SET pgfs.allowed_local_fs_paths = '{{allowed_paths}}'
    print("Allowed after the restart:", allowed_paths)

## Restart the database

This is required for the new settings to start working, both the file-system allow-list and the MCP server settings from earlier. The command performs a fast restart of Postgres, which closes the notebook's database connections. Wait a few seconds for it to finish before running the next cell.

In [ ]:
!/usr/pgsql-18/bin/pg_ctl -D /var/lib/pgsql/18/data restart -m fast

## Connect after restart

Restarting Postgres dropped the earlier connections, so create a new engine and re-attach the `%sql` magics before continuing.

In [ ]:
conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}",
    pool_pre_ping=True
)

%reload_ext sql
%sql conn

## MCP: Show information after restart

List the MCP settings as Postgres now sees them. You should see the values configured earlier, and `pending_restart` should be `false` for each of them, which confirms that the restart picked them up.

In [ ]:
%%sql
SELECT name, setting, pending_restart
FROM pg_settings
WHERE name LIKE 'edb.endpoints_mcp%';

## MCP: Verify ready

Call the MCP server's `/health` and `/ready` endpoints. Both should return a success response. `--insecure` is needed because the certificate is self-signed.

If either call fails, check that the settings above show `pending_restart = false`, and that port 8765 is allowed through any firewall or AWS security group.

In [ ]:
!curl -fsS --insecure https://{MCP_HOST}:8765/health
!echo ""
!curl -fsS --insecure https://{MCP_HOST}:8765/ready

## Knowledge Base: Create PGFS storage location

A **storage location** gives a name (`mcp_pdfs`) to a place where files live. This example uses local storage, but it also supports S3.

In [ ]:
%%sql
SELECT pgfs.create_storage_location('mcp_pdfs', 'file:///var/lib/pgsql/pgfs/');

## Knowledge Base: Create AIDB volume

A **volume** presents the files in a storage location to AIDB as a source that pipelines can read. This one points at the `pdfs/` folder and treats its files as PDF documents. The Volume will be the source for an AI Pipeline.

The first statement deletes any earlier volume on this storage location (`mcp_pdfs`) so the cell can be re-run. Volumes on other storage locations are left alone.

In [ ]:
%%sql
SELECT aidb.delete_volume(schema || '.' || volume) FROM aidb.volumes WHERE storage_location = 'mcp_pdfs';
SELECT aidb.create_volume('mcp_pdf_volume', 'mcp_pdfs', 'pdfs/', 'Pdf');

## Knowledge Base: Download demo files

These are 2025 IRS tax documents, downloaded directly into the folder registered for the storage location:

- **`i1040gi.pdf`**: the instructions for Form 1040.
- **`p17.pdf`**: Publication 17, *Your Federal Income Tax*.

The download runs on the database host, so it needs internet access. If the host is offline, copy the two files into the same folder manually instead.

In [ ]:
!curl https://www.irs.gov/pub/irs-pdf/i1040gi.pdf > /var/lib/pgsql/pgfs/pdfs/i1040gi.pdf
!curl https://www.irs.gov/pub/irs-pdf/p17.pdf > /var/lib/pgsql/pgfs/pdfs/p17.pdf

## Knowledge Base: AI pipeline

AI Pipelines can ingest, prepare, and vectorize data for semantic retrieval and RAG applications. They integrate pgvector search capabilities and can keep the knowledge base synchronized with your source data automatically.

This pipeline has three steps:

1. **`ParsePdf`** extracts the text from each PDF. `Structured` parsing preserves layout such as tables, and `allow_partial_parsing` lets the pipeline continue if part of a file cannot be read.
2. **`ChunkText`** splits the text into pieces of about 100 words (never more than 120), with 10 words of overlap between neighbouring chunks. This improves the accuracy of searches in large documents.
3. **`KnowledgeBase`** creates a vector embedding for each chunk with the `bge-small-en-v1.5-f16` model.

`aidb.run_pipeline` then runs it, which can take a few minutes because the PDFs are large.

The last three statements are the **governance** part of this step. Since the tool will run as `mcp_agent`, that role is granted `USAGE` on the `public` schema and `SELECT` on only the two tables the search needs: the knowledge base (`pipeline_mcp_pdf_kb`) and the chunk text (`pipeline_mcp_pdf_kb_step_2`). Nothing else in `public` is exposed to Claude.

In [ ]:
%%sql
SELECT aidb.delete_pipeline(name)
FROM aidb.pipelines
WHERE name = 'mcp_pdf_kb';

SELECT aidb.create_pipeline(
    name           => 'mcp_pdf_kb',
    source         => 'mcp_pdf_volume',
    step_1         => 'ParsePdf',
    step_1_options => aidb.pdf_parse_config(method => 'Structured', allow_partial_parsing => true),
    step_2         => 'ChunkText',
    step_2_options => aidb.chunk_text_config(desired_length => 100, max_length => 120, overlap_length => 10, strategy => 'words')
                      || jsonb_build_object('intermediate_destination', jsonb_build_object('enabled', true)),
    step_3         => 'KnowledgeBase',
    step_3_options => aidb.knowledge_base_config(model => 'bge-small-en-v1.5-f16', data_format => 'Text'),
    auto_processing => 'Disabled'
);

SELECT aidb.run_pipeline('mcp_pdf_kb');

GRANT USAGE ON SCHEMA public TO mcp_agent;
GRANT SELECT ON TABLE public.pipeline_mcp_pdf_kb TO mcp_agent;
GRANT SELECT ON TABLE public.pipeline_mcp_pdf_kb_step_2 TO mcp_agent;

## Knowledge Base: Example question

We can execute similarity searches on the vector embeddings in the database, but the real power is when you create a SQL tool and use an agent to ask questions about your data.

This query asks *"what are the new rules for taxes on tips?"* and shows the top three matching chunks. It confirms the knowledge base works before Claude Desktop is involved.

In [ ]:
%%sql
SELECT key,
       part_ids,
       distance::numeric(5,3) AS dist,
       intermediate_steps -> 0 ->> 'value' AS chunk_text
FROM aidb.retrieve_text('public.pipeline_mcp_pdf_kb', -- knowledgebase 
                        'what are the new rules for taxes on tips?', --question
                        8, --distance
                        deduplicate => false)
ORDER BY 3 DESC
LIMIT 3;

## Knowledge Base: SQL tool

The SQL tool can be used by agents either running inside Postgres or externally, like Claude Desktop. For this demo, we will use Claude Desktop via an MCP server.

- `description` is what Claude reads to decide *when* to use the tool, so it names the documents it covers and when to use it.
- `${query}` and `${top_k}` are placeholders Claude fills in. They are declared in `params` with a type and a description.
- `read_only => true` marks the tool as safe because it only reads data.

Once it exists, the AIDB MCP endpoint lists `mcp_search_tax_docs` among the tools it offers to clients.

In [ ]:
%%sql
SELECT aidb.delete_tool(name)
FROM aidb.sql_tool_registry
WHERE name = 'mcp_search_tax_docs';

SELECT aidb.create_sql_tool(
    name          => 'mcp_search_tax_docs',
    description   => 'Semantic search over IRS 2025 Form 1040 instructions and Publication 17. Returns the most relevant passages with their source file. Use for any tax rules question.',
    sql_statement => $$
        SELECT key AS source_file,
               distance::numeric(5,3) AS distance,
               intermediate_steps -> 0 ->> 'value' AS passage
        FROM aidb.retrieve_text('public.pipeline_mcp_pdf_kb'::text, ${query}::text, ${top_k}::int, deduplicate => false)
    $$,
    params        => aidb.tool_params(
        aidb.tool_param('query', 'TEXT', 'The question or search phrase.'),
        aidb.tool_param('top_k', 'INT', 'Number of passages to return, e.g. 6.')
    ),
    read_only     => true
);

## Claude Desktop: Create shell script

Claude Desktop starts MCP servers as local programs that it talks to over standard input and output, but our server is remote and uses HTTPS with a login. A small **bridge script** connects the two. It:

1. Reads each message from Claude Desktop, one per line.
2. Sends it to the MCP server's URL with `curl`, including the login and the session ID.
3. Remembers the session ID the server returns, and sends it with later requests.
4. Converts the server's streamed response back into the plain lines Claude Desktop expects.

For demo purposes this uses basic authentication over an HTTPS request to the MCP server, and `curl -k` skips certificate verification because the certificate is self-signed.

First, run this cell and then click on the **Download** link below it. The script is generated in your browser, so nothing is saved on the server.

In [ ]:
import base64
from IPython.display import HTML

SCRIPT = r'''#!/bin/bash
SESSION=""
hdr=$(mktemp)
trap 'rm -f "$hdr"' EXIT

while IFS= read -r line; do
  [ -z "$line" ] && continue

  args=(-ksS --http1.1 -D "$hdr" -X POST "$MCP_URL"
        -H "Host: localhost:8765"
        -H "Content-Type: application/json"
        -H "Accept: application/json, text/event-stream"
        -H "Authorization: $MCP_AUTH"
        --data-binary "$line")
  [ -n "$SESSION" ] && args+=(-H "mcp-session-id: $SESSION")

  body=$(/usr/bin/curl "${args[@]}")

  s=$(awk -F': ' 'tolower($1)=="mcp-session-id"{gsub("\r","",$2);print $2}' "$hdr")
  [ -n "$s" ] && SESSION="$s"

  out=$(printf '%s\n' "$body" | sed -n 's/^data: *//p' | grep -v '^$')
  [ -z "$out" ] && out="$body"
  [ -n "$out" ] && printf '%s\n' "$out"
done
'''

b64 = base64.b64encode(SCRIPT.encode()).decode()
display(HTML(f'<a download="mcp-curl-bridge.sh" href="data:text/plain;base64,{b64}">Download mcp-curl-bridge.sh</a>'))

## Claude Desktop: Move script

Place the downloaded `mcp-curl-bridge.sh` file in your `~/.local/bin/` directory. Create the directory first if it does not exist. Claude's configuration runs the script through `bash`, so you do not need to make it executable.

```
mkdir -p ~/.local/bin
mv ~/Downloads/mcp-curl-bridge.sh ~/.local/bin
```

## Claude Desktop: Update developer config
Execute this cell and in Claude Desktop, go to Settings → Developer → Edit Config. That opens the folder containing `claude_desktop_config.json`.

The cell prints a JSON block that defines one MCP server named `edb-aidb`:

- `command` and `args` tell Claude Desktop to start the bridge script.
- `MCP_URL` is the address of your MCP server (`https://<host>:8765/mcp`).
- `MCP_AUTH` is the `mcp_agent` role's login, encoded for HTTP basic authentication.

Paste the JSON from your notebook into the file. How you do that depends on what’s already in it.

- File is empty or just {}: replace the contents with the whole JSON output.
- File already has an "mcpServers" block: don’t paste the whole output. Copy only the "edb-aidb": { ... } entry and add it inside the existing mcpServers, with a comma between entries.
- File has other settings but no mcpServers: add "mcpServers": { ... } from the output alongside them, with a comma between top-level keys.

After saving the file, fully quit and reopen Claude Desktop. Closing the window isn’t enough:
- Mac: press Cmd+Q, or choose Claude → Quit Claude, then reopen it.
- Windows: right-click the Claude icon in the system tray, choose Quit, then reopen it.

*The output contains the encoded password, so treat it as a secret and do not share it.*

In [ ]:
import base64, json

auth = base64.b64encode(f"mcp_agent:{MCP_PASSWORD}".encode()).decode()

config = {
    "mcpServers": {
        "edb-aidb": {
            "command": "/bin/bash",
            "args": ["-c", "exec /bin/bash \"$HOME/.local/bin/mcp-curl-bridge.sh\""],
            "env": {
                "MCP_URL": f"https://{MCP_HOST}:8765/mcp",
                "MCP_AUTH": f"Basic {auth}"
            }
        }
    }
}

print(json.dumps(config, indent=2))

## Validate

After restarting Claude Desktop, open a new chat and provide this prompt:

```
Use the mcp_search_tax_docs tool to find the 2025 standard deduction, and tell me the source files.

```

**What to expect:** Claude Desktop may ask for permission to use the `mcp_search_tax_docs` tool from `edb-aidb`. Once you allow it, Claude calls the tool, receives passages from the IRS PDFs, and answers using them. Because the tool returns the source file with every passage, the answer can name `i1040gi.pdf` and `p17.pdf`, so you can check where each fact came from.

**If the tool does not appear or fails**

- Make sure you fully quit and reopened Claude Desktop, and that the JSON in `claude_desktop_config.json` is valid.
- Check that the bridge script is in `~/.local/bin/` and that the MCP server is reachable from your Mac (the `/health` check earlier, and the security group on AWS).
- In Claude Desktop, open Settings → Developer to see whether the `edb-aidb` server is running and view its logs.
- Confirm the password in `MCP_PASSWORD` matches what you set when creating the `mcp_agent` role.

**Before using this beyond a demo**

- Replace the placeholder password with a strong secret.
- Use a certificate from a trusted certificate authority and turn certificate verification back on.
- Limit who can reach the MCP port, and narrow the `mcp_agent` grants to only what is needed.